# BOAMP Reference Benchmark Preparation

## tl;dr

Executed successfully. The manual benchmark was materialized as `120` reference anchors, `94` primary evaluation anchors, and `28` confirmed successor-link rows. All anchor notice IDs and successor evidence IDs map back to the Grand Ouest BOAMP data. `PILOT_DEVELOPMENT` and `LOCKED_TEST` are preserved for tuning and final evaluation.

## Context & Methods

The reference files are treated as trusted manual evidence. They are not used to generate model features, except to define benchmark anchors and truth labels for evaluation. `PILOT_DEVELOPMENT` is for tuning and `LOCKED_TEST` is for final evaluation.


In [1]:
from __future__ import annotations

import ast
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/reference/BOAMP_Internship_Reference_120.csv").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data/reference"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp_grand_ouest"
NOTICES_PATH = PROCESSED_DIR / "notices_engineered.parquet"
ANCHOR_OUTPUT = PROCESSED_DIR / "reference_anchor_episodes.parquet"
LINK_OUTPUT = PROCESSED_DIR / "reference_successor_links.parquet"
SUMMARY_OUTPUT = PROCESSED_DIR / "reference_quality_summary.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_120_PATH = REFERENCE_DIR / "BOAMP_Internship_Reference_120.csv"
EVALUATION_SUBSET_PATH = REFERENCE_DIR / "BOAMP_Internship_Evaluation_Subset.csv"
CONFIRMED_LINKS_PATH = REFERENCE_DIR / "BOAMP_Internship_Confirmed_Successor_Links.csv"

IDWEB_URL_RE = re.compile(r"idweb:([0-9]{2}-[0-9]+)")
print(PROJECT_ROOT)


/home/senghakrou/project-gigalis


## Data

### 1. Load Reference Files And Helpers


In [2]:
def parse_json_list(value) -> list:
    if value is None or pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(text)
        except Exception:
            return []
    return parsed if isinstance(parsed, list) else [parsed]


def idweb_from_url(url: str) -> str | None:
    match = IDWEB_URL_RE.search(str(url))
    return match.group(1) if match else None


def idwebs_from_urls_json(value) -> list[str]:
    ids = []
    for url in parse_json_list(value):
        idweb = idweb_from_url(url)
        if idweb and idweb not in ids:
            ids.append(idweb)
    return ids

reference_120 = pd.read_csv(REFERENCE_120_PATH, dtype="string", keep_default_na=False)
evaluation_subset = pd.read_csv(EVALUATION_SUBSET_PATH, dtype="string", keep_default_na=False)
confirmed_links = pd.read_csv(CONFIRMED_LINKS_PATH, dtype="string", keep_default_na=False)
notices = pd.read_parquet(NOTICES_PATH, columns=["idweb", "dateparution", "buyer_name_raw", "buyer_name_normalized", "grand_ouest_region", "primary_cpv", "cpv_codes_json", "objet_normalized", "url_avis"])
notice_ids = set(notices["idweb"].astype(str))

print(reference_120.shape, evaluation_subset.shape, confirmed_links.shape, notices.shape)


(120, 49) (94, 17) (28, 13) (226314, 9)


### 2. Build Anchor Episode Benchmark Table


In [3]:
anchors = reference_120.copy()
anchors["is_in_evaluation_subset"] = anchors["sample_id"].isin(set(evaluation_subset["sample_id"]))
anchors["anchor_notice_ids_list"] = anchors["anchor_notice_ids_json"].map(parse_json_list)
anchors["anchor_notice_ids_count"] = anchors["anchor_notice_ids_list"].map(len).astype("int64")
anchors["anchor_notice_ids_found_count"] = anchors["anchor_notice_ids_list"].map(lambda values: sum(str(v) in notice_ids for v in values)).astype("int64")
anchors["anchor_notice_ids_missing_json"] = anchors["anchor_notice_ids_list"].map(lambda values: json.dumps([str(v) for v in values if str(v) not in notice_ids], ensure_ascii=False))
anchors["anchor_notice_ids_list_json"] = anchors["anchor_notice_ids_list"].map(lambda values: json.dumps([str(v) for v in values], ensure_ascii=False))
anchors["anchor_regions_list_json"] = anchors["anchor_regions_json"].map(lambda value: json.dumps(parse_json_list(value), ensure_ascii=False))
anchors["anchor_cpv_codes_list_json"] = anchors["anchor_cpv_codes_json"].map(lambda value: json.dumps(parse_json_list(value), ensure_ascii=False))
anchors["final_successor_episode_ids_list_json"] = anchors["final_successor_episode_ids"].map(lambda value: json.dumps(parse_json_list(value), ensure_ascii=False))
anchors["final_evidence_idwebs_json"] = anchors["final_evidence_urls"].map(lambda value: json.dumps(idwebs_from_urls_json(value), ensure_ascii=False))

anchor_keep = [
    "sample_id", "benchmark_split", "sampling_stratum", "sampling_weight", "anchor_episode_id",
    "anchor_notice_ids_list_json", "anchor_notice_ids_count", "anchor_notice_ids_found_count", "anchor_notice_ids_missing_json",
    "anchor_award_notice_date", "anchor_expected_end_date", "anchor_buyer_siren", "anchor_buyer_name",
    "anchor_regions_list_json", "anchor_cpv_codes_list_json", "anchor_theme", "anchor_duration_months", "anchor_text",
    "broad_candidate_pool_n", "review_candidates_exported_n", "final_outcome", "final_successor_episode_ids_list_json",
    "final_confidence", "final_evidence_idwebs_json", "final_reason", "primary_evaluation_eligible", "quality_flag",
    "exclusion_reason", "review_method", "source_coverage", "review_date", "reference_version", "is_in_evaluation_subset",
]
anchors_out = anchors[anchor_keep].copy()
anchors_out.to_parquet(ANCHOR_OUTPUT, index=False, compression="zstd")
display(anchors_out.head())


,sample_id,benchmark_split,sampling_stratum,sampling_weight,anchor_episode_id,anchor_notice_ids_list_json,anchor_notice_ids_count,anchor_notice_ids_found_count,anchor_notice_ids_missing_json,anchor_award_notice_date,anchor_expected_end_date,anchor_buyer_siren,anchor_buyer_name,anchor_regions_list_json,anchor_cpv_codes_list_json,anchor_theme,anchor_duration_months,anchor_text,broad_candidate_pool_n,review_candidates_exported_n,final_outcome,final_successor_episode_ids_list_json,final_confidence,final_evidence_idwebs_json,final_reason,primary_evaluation_eligible,quality_flag,exclusion_reason,review_method,source_coverage,review_date,reference_version,is_in_evaluation_subset
0,RV-001,PILOT_DEVELOPMENT,CPV-32|buyer_id|duration,4.0,EP-95499bedb1e980f8,"[""18-146639"", ""19-67124""]",2,2,[],2019-04-28,2020-04-28,247200132.0,Le Mans Métropole - Communauté Urbaine,"[""Pays de la Loire""]","[""39300000"", ""43800000"", ""45259000"", ""42512500"", ""34322100"", ""31600000"", ""32236000"", ""30200000"", ""42961100""]",CPV-32,12.0,acquisition de materiels divers pour la setram les prestations sont reparties en 15 lots les variantes ne sont pas autorisees sauf pour le lot no11 variante exigee pour doubler...,501,25,OUTSIDE_SCOPE,[],HIGH,"[""18-146639"", ""19-67124""]",Mixed SETRAM equipment procurement dominated by general operational equipment rather than a clearly defined digital contract.,False,,Anchor is outside the intended digital-procurement scope.,SINGLE_EVIDENCE_REVIEW_SUPPLIED_BOAMP_CORPUS,BOAMP supplied corpus and official notice URLs; no exhaustive TED/DECP search,2026-08-11,internship_v1.0,False
1,RV-002,PILOT_DEVELOPMENT,CPV-48|buyer_id|duration_missing,2.0,EP-196eb8476722f479,"[""18-152232"", ""20-49689""]",2,2,[],2020-04-09,,781107446.0,HABITAT 76,"[""Normandie""]","[""48781000"", ""50324100"", ""72230000""]",CPV-48,,renouvellement du systeme d information ressources humaines en mode saas modules paies gestion administrative gestion des temps et des activites renouvellement du systeme d inf...,40,25,NO_OBSERVED_SUCCESSOR_IN_SCOPE,[],MEDIUM,"[""18-152232"", ""20-49689"", ""21-16150"", ""21-112471"", ""21-75704"", ""22-50332"", ""21-40468"", ""21-90944""]",No later episode in the supplied BOAMP review candidates matched both the buyer continuity and the anchor's specific functional need. The reviewed candidates concerned material...,True,,,SINGLE_EVIDENCE_REVIEW_SUPPLIED_BOAMP_CORPUS,BOAMP supplied corpus and official notice URLs; no exhaustive TED/DECP search,2026-08-11,internship_v1.0,True
2,RV-003,PILOT_DEVELOPMENT,CPV-32|buyer_id|duration,4.0,EP-942e1a0a861bb53e,"[""21-137784"", ""22-19361""]",2,2,[],2022-02-09,2023-02-10,200069839.0,Cté de Cnes de la Côte d'Albatre,"[""Normandie""]","[""37300000"", ""37313000"", ""37314000"", ""37312000"", ""37311000"", ""37316000"", ""37321500"", ""32342410"", ""37315000"", ""37000000"", ""37313300"", ""37480000""]",CPV-32,12.0,acquisition d instruments et materiels de musique le conservatoire musique et danse de la cote d albatre est un etablissement classe par l etat a rayonnement intercommunal ayan...,74,25,OUTSIDE_SCOPE,[],HIGH,"[""21-137784"", ""22-19361""]",Musical instruments and music equipment are not part of the intended public-sector digital procurement scope.,False,,Anchor is outside the intended digital-procurement scope.,SINGLE_EVIDENCE_REVIEW_SUPPLIED_BOAMP_CORPUS,BOAMP supplied corpus and official notice URLs; no exhaustive TED/DECP search,2026-08-11,internship_v1.0,False
3,RV-004,PILOT_DEVELOPMENT,CPV-72|buyer_id|duration,27.714285714285715,EP-a0a604b69ebe8a9d,"[""16-111867"", ""17-6770""]",2,2,[],2017-01-18,2023-01-19,224900019.0,Département de Maine-et-Loire,"[""Pays de la Loire""]","[""72210000"", ""72267000""]",CPV-72,72.0,integration du systeme d archivage electronique as lae sur une infrastructure mutualisee integration du systeme d archivage electronique as lae sur une infrastructure mutualise...,685,25,NO_OBSERVED_SUCCESSOR_IN_SCOPE,[],MEDIUM,"[""16-111867"", ""

### 3. Build Link-Level Positive Truth Table


In [4]:
links = confirmed_links.copy()
links["successor_notice_ids_list"] = links["candidate_urls_json"].map(idwebs_from_urls_json)
links["successor_notice_ids_json"] = links["successor_notice_ids_list"].map(lambda values: json.dumps(values, ensure_ascii=False))
links["successor_notice_ids_count"] = links["successor_notice_ids_list"].map(len).astype("int64")
links["successor_notice_ids_found_count"] = links["successor_notice_ids_list"].map(lambda values: sum(v in notice_ids for v in values)).astype("int64")
links["successor_notice_ids_missing_json"] = links["successor_notice_ids_list"].map(lambda values: json.dumps([v for v in values if v not in notice_ids], ensure_ascii=False))
links["publication_gap_days"] = pd.to_numeric(links["publication_gap_days"], errors="coerce").astype("Int64")
links["is_primary_evaluation_eligible_anchor"] = links["sample_id"].isin(set(anchors_out.loc[anchors_out["primary_evaluation_eligible"].eq("True"), "sample_id"]))

link_keep = [
    "sample_id", "anchor_episode_id", "successor_episode_id", "final_outcome", "final_confidence",
    "anchor_award_notice_date", "candidate_start_date", "publication_gap_days", "anchor_buyer_name", "candidate_buyer_name",
    "anchor_theme", "successor_notice_ids_json", "successor_notice_ids_count", "successor_notice_ids_found_count",
    "successor_notice_ids_missing_json", "final_reason", "is_primary_evaluation_eligible_anchor",
]
links_out = links[link_keep].copy()
links_out.to_parquet(LINK_OUTPUT, index=False, compression="zstd")
display(links_out.head())


,sample_id,anchor_episode_id,successor_episode_id,final_outcome,final_confidence,anchor_award_notice_date,candidate_start_date,publication_gap_days,anchor_buyer_name,candidate_buyer_name,anchor_theme,successor_notice_ids_json,successor_notice_ids_count,successor_notice_ids_found_count,successor_notice_ids_missing_json,final_reason,is_primary_evaluation_eligible_anchor
0,RV-010,EP-a547673afe2dda78,EP-30ed99860ba06292,OBSERVED_SUCCESSOR,MEDIUM,2020-10-11,2025-03-11,1612,Université de Bretagne Occidentale,Université de Bretagne Occidentale,CPV-32,"[""25-26886""]",1,1,[],Same UBO buyer and a later procurement to refit the university computer networks; the functional network need continues in a new procedure.,True
1,RV-011,EP-761e9e86bf100636,EP-9fdcd915e2532264,OBSERVED_SUCCESSOR,HIGH,2017-07-14,2021-03-28,1353,Région Bretagne,Région Bretagne,CPV-32,"[""21-40411"", ""21-113756""]",2,2,[],Same Région Bretagne buyer and a later telecommunications framework covering the same fixed and mobile telecommunications need.,True
2,RV-012,EP-be2c7634a4783b45,EP-c2d565c3cbc7ef2c,OBSERVED_SUCCESSOR,HIGH,2017-04-05,2021-03-19,1444,Conseil régional des Pays de la Loire,Conseil régional des Pays de la Loire,CPV-72,"[""21-35651"", ""21-52670""]",2,2,[],Same regional buyer and a later contract explicitly continuing support for execution of the Pays de la Loire digital master plan.,True
3,RV-015,EP-0c7a42dbe93a0197,EP-7580adec0b1d9f9f,OBSERVED_SUCCESSOR,MEDIUM,2015-12-31,2016-10-17,291,Brest métropole,Brest métropole,CPV-32,"[""16-148531"", ""16-184860""]",2,2,[],"Same Brest buyer and a new works procedure continuing cabling for traffic signalling, urban equipment and the high-speed network.",True
4,RV-019,EP-88c514f9a822db41,EP-658043f693108dc1,OBSERVED_SUCCESSOR,HIGH,2019-11-27,2023-03-30,1219,Syndicat Mixte Manche Numérique,Syndicat Mixte Manche Numérique,CPV-32,"[""23-41674"", ""23-56852"", ""23-101859""]",3,3,[],"Same Manche Numérique buyer and a later framework covering mobile services, devices and accessories, continuing the mobile-telephony purchasing need.",True


## Results

### 4. Validate Reference Consistency


In [5]:
json_columns = {
    "reference_120": ["anchor_notice_ids_json", "anchor_regions_json", "anchor_cpv_codes_json", "anchor_urls_json", "final_successor_episode_ids", "final_evidence_urls"],
    "evaluation_subset": ["final_successor_episode_ids", "final_evidence_urls"],
    "confirmed_links": ["candidate_urls_json"],
}
json_errors = {}
for name, frame in [("reference_120", reference_120), ("evaluation_subset", evaluation_subset), ("confirmed_links", confirmed_links)]:
    for column in json_columns[name]:
        bad = 0
        for value in frame[column]:
            try:
                parse_json_list(value)
            except Exception:
                bad += 1
        json_errors[f"{name}.{column}"] = bad

summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "inputs": {"reference_120": str(REFERENCE_120_PATH), "evaluation_subset": str(EVALUATION_SUBSET_PATH), "confirmed_links": str(CONFIRMED_LINKS_PATH)},
    "outputs": {"reference_anchor_episodes": str(ANCHOR_OUTPUT), "reference_successor_links": str(LINK_OUTPUT)},
    "row_counts": {"reference_120": int(len(reference_120)), "evaluation_subset": int(len(evaluation_subset)), "confirmed_links": int(len(confirmed_links))},
    "outcomes_reference_120": {str(k): int(v) for k, v in reference_120["final_outcome"].value_counts().items()},
    "outcomes_evaluation_subset": {str(k): int(v) for k, v in evaluation_subset["final_outcome"].value_counts().items()},
    "splits_evaluation_subset": {str(k): int(v) for k, v in evaluation_subset["benchmark_split"].value_counts().items()},
    "coverage": {
        "evaluation_sample_ids_subset_of_reference": bool(set(evaluation_subset["sample_id"]).issubset(set(reference_120["sample_id"]))),
        "confirmed_link_sample_ids_subset_of_reference": bool(set(confirmed_links["sample_id"]).issubset(set(reference_120["sample_id"]))),
        "anchor_notice_missing_total": int(sum(len(json.loads(v)) for v in anchors_out["anchor_notice_ids_missing_json"])),
        "successor_notice_missing_total": int(sum(len(json.loads(v)) for v in links_out["successor_notice_ids_missing_json"])),
    },
    "json_parse_errors": json_errors,
    "notes": {
        "primary_evaluation_subset": "Rows where primary_evaluation_eligible is True.",
        "multiple_successors_supported": True,
        "locked_test_not_for_tuning": True,
    },
}
SUMMARY_OUTPUT.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

checks = pd.DataFrame([
    ["Reference rows", len(reference_120)],
    ["Evaluation rows", len(evaluation_subset)],
    ["Confirmed link rows", len(confirmed_links)],
    ["Anchor notice IDs missing", summary["coverage"]["anchor_notice_missing_total"]],
    ["Successor notice IDs missing", summary["coverage"]["successor_notice_missing_total"]],
    ["Evaluation subset of reference", summary["coverage"]["evaluation_sample_ids_subset_of_reference"]],
], columns=["check", "value"])
display(checks)
display(pd.DataFrame(summary["outcomes_evaluation_subset"].items(), columns=["outcome", "rows"]))
assert summary["coverage"]["anchor_notice_missing_total"] == 0
assert summary["coverage"]["successor_notice_missing_total"] == 0
assert summary["coverage"]["evaluation_sample_ids_subset_of_reference"]


,check,value
0,Reference rows,120
1,Evaluation rows,94
2,Confirmed link rows,28
3,Anchor notice IDs missing,0
4,Successor notice IDs missing,0
5,Evaluation subset of reference,True


,outcome,rows
0,NO_OBSERVED_SUCCESSOR_IN_SCOPE,70
1,OBSERVED_SUCCESSOR,23
2,MULTIPLE_SUCCESSORS,1


## Takeaways

The reference benchmark is now available as episode-level anchor rows and link-level positive truth rows. The primary benchmark remains the 94 eligible anchors, with pilot and locked-test splits preserved.
